In [18]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

Traceback (most recent call last):
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-p

In [ ]:
MODEL = 'llama3.2'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

Traceback (most recent call last):
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pradeepkumar/miniconda3/envs/llms/lib/python3.11/site-p

In [5]:
links = fetch_website_links("https://anthropic.com")

links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://docs.claude.com',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/product/cowork',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 'https://www.anthropic.com/claude/sonnet',
 'https://www

## Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. 

In [8]:
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to an About page, or a Company page, or Careers/Jobs pages.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    }
    """

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    
    return user_prompt

In [9]:
print(get_links_user_prompt("https://anthropic.com"))


        Here is the list of links on the website https://anthropic.com -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    #main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/economic-futures
https://www.anthropic.com/constitution
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
http://trust.anthropic.com/
https://www.anthropic.com/learn
https://claude.com/resources/tutorials
https://claude.com/resources/use-cases
https://www.anthropic.com/engineering
https://docs.claude.com
https://www.anthropic.com/company
https://www.anthropic.com/careers
https://www.anthropic.com/events
https://www.anthropic.com/news
https://claude.ai
https://claude.com/product/overview
https://claude.com/produc

In [20]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [21]:
select_relevant_links("https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 11 relevant links


{'links': [{'type': 'Company page', 'url': 'https://www.anthropic.com/'},
  {'type': 'About us page', 'url': 'https://docs.claude.com'},
  {'type': 'Transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'Responsible scaling policy',
   'url': 'https://www.anthropic.com/responsible-scaling-policy'},
  {'type': 'Research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'Economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': 'Constitution page',
   'url': 'https://www.anthropic.com/constitution'},
  {'type': 'Careers página', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'About us blog', 'url': 'https://www.claude.com/blog'},
  {'type': 'Community page', 'url': 'https://claude.com/community'},
  {'type': 'Learn section home page',
   'url': 'https://www.anthropic.com/learn'}]}

# Let's make a brochure now!

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://anthropic.com"))

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 13 relevant links
## Landing Page:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
Products
Claude
Claude Code
Claude Cowork
Claude Platform
Pricing
Contact sales
Models
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Develope

In [ ]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [34]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [35]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream = True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [36]:
stream_brochure("Antropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 20 relevant links


# Anthropic: Securing the Benefits and Mitigating the Risks of AI

## About Us

Anthropic is a public benefit corporation dedicated to harnessing the power of artificial intelligence (AI) while prioritizing safety and responsible growth. Our mission is to ensure that the benefits of AI are realized, while minimizing its risks.

## Research and Initiatives

At Anthropic, we're committed to advancing our understanding of AI's impact on society through research and initiatives. This includes studies like the largest ever conducted on how AI is shaping lives around the world, as well as collaborations with policymakers and stakeholders to ensure that AI is developed and deployed responsibly.

## Our Models

Our flagship model, Claude Opus 4.6, is the world's most powerful model for coding, agents, and professional work. This cutting-edge technology enables developers, researchers, and businesses to create more sophisticated, interactive, and human-like applications.

## Products and Tools

We offer a suite of products and tools designed to facilitate safe and effective AI development:

* Claude Code: A platform for developing intelligent systems that align with human values.
* Claude Cowork: A space for meaningful conversations without ads or sponsored content.
* Claude Platform: Our comprehensive platform for building, deploying, and managing AI applications.

## Join the Anthropic Community

Be part of our mission to secure AI's benefits and mitigate its risks. Explore our resources, download our models, and join our community to stay up-to-date on the latest developments in AI research and innovation.

## Careers

We're seeking talented individuals to join our team of experts in AI development, research, and policy. Check out our job listings and learn more about our company culture, which values collaboration, inclusivity, and professionalism.

## Website broucher with Gradio UI

In [5]:
import gradio as gr

In [3]:
model_llama = 'llama3.2'
model_phi = 'phi3'

In [6]:
system_message = "You are an assistant that analyzes the contents of a company website landing page and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown without code blocks."

In [33]:
def stream_llama(prompt):
    message = [
        {"role" : "system", "content" : system_message},
        {"role" : "user", "content" : prompt}
    ]

    stream = openai.chat.completions.create(model = model_llama, messages = message, stream = True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

def stream_phi(prompt):
    message = [
        {"role" : "system", "content" : system_message},
        {"role" : "user", "content" : prompt}
    ]

    stream = openai.chat.completions.create(model = model_phi, messages = message, stream = True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [34]:
def stream_brochure(company_name, url, model):
    prompt = f"Please generate a company brochure for {company_name}. Here are their landing page:\n"
    prompt += fetch_website_contents(url)

    if model == "Model llama3.2":
        result = stream_llama(prompt)
    if model == "Model Phi3":
        result = stream_phi(prompt)

    yield from result



In [36]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["Model llama3.2", "Model Phi3"], label="Select model", value="Model llama3.2")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "Model llama3.2"],
            ["Anthropic", "https://anthropic.com", "Model Phi3"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
